# Whisper 语音转文字

[English](01_finetune.ipynb) | 简体中文

## Goal / 目标

使用与 CLI 相同的实现，完成音频校验、epoch 边界恢复、held-out 评估、导出重载与推理。默认执行离线 CPU smoke，不代表真实语音质量。

> Smoke 使用随机初始化原生 Whisper 与合成 tokenizer。正弦音不包含对应文字。

## Setup / 环境准备

语音使用独立环境：Transformers 4.57.3 不能与固定 5.18.0 的任务混装。下一格仅在 Colab 安装依赖，本地需预先安装对应 extra。若替换已加载依赖请重启 runtime；长期实验使用 Drive 持久化。

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    ROOT = Path("/content/advai-notebooks")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/advai-x/advai-notebooks.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}[stt,notebooks]"], check=True)
else:
    ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                if (path / "pyproject.toml").exists())
print("Repository and speech environment ready.")

Repository and speech environment ready.


### 导入与参数

Seed、数据、预算与 optimizer 来自仓库 YAML；每次新建输出目录，支持完整重跑。

In [2]:
import json
import uuid
import warnings
import torch
from IPython.display import Audio, Markdown, display
from advai_notebooks.stt.config import load_config
from advai_notebooks.stt.data import load_data, prepare_data
from advai_notebooks.stt.model import STTModel
from advai_notebooks.stt.pipeline import run
from advai_notebooks.stt.publish import publish
from advai_notebooks.common.runtime import setup
from transformers.utils import logging

logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="Moving the following attributes.*")
cfg = load_config(ROOT / "configs/stt/smoke.yaml", ROOT)
cfg.output_dir = f"outputs/stt/notebook-{uuid.uuid4().hex[:8]}"
print(json.dumps({"model": cfg.model.id, "seed": cfg.training.seed,
                  "epochs": cfg.training.epochs, "device": cfg.runtime.device}, indent=2))

{
  "model": "tiny",
  "seed": 42,
  "epochs": 2,
  "device": "cpu"
}


## Steps / 步骤

### 1. 校验与准备音频

Group 和相同解码音频不能跨 split；片段须为有限单声道且满足时长。检查有限 manifest 样例并试听合成输入。

In [3]:
rows, summary = load_data(cfg)
validator = STTModel(cfg.model, setup(cfg), cfg.root, cfg.training)
prepared = prepare_data(rows, validator, cfg, fit=True)
print(json.dumps({"splits": summary["splits"], "sample_rate": summary["sample_rate"]}, indent=2))
for row in rows[:2]:
    print({key: row[key] for key in ("id", "group_id", "split", "text")})
display(Audio(rows[0]["waveform"], rate=summary["sample_rate"]))
del validator, prepared

{
  "splits": {
    "train": 4,
    "validation": 2,
    "test": 2
  },
  "sample_rate": 16000
}
{'id': 'one', 'group_id': 'recording-1', 'split': 'train', 'text': 'red'}
{'id': 'two', 'group_id': 'recording-2', 'split': 'train', 'text': 'green'}


### 2. 训练并在 epoch one 后模拟中断

Pipeline 在调用 callback 前保存基线与原子 checkpoint，模拟 epoch 边界断连，同时保留 optimizer/RNG 状态。

In [4]:
class TutorialInterruption(Exception):
    pass

def interrupt_after_first_epoch(epoch, state):
    if epoch == 1:
        raise TutorialInterruption("Saved epoch-one checkpoint.")

try:
    run(cfg, callbacks=(interrupt_after_first_epoch,))
except TutorialInterruption as error:
    print(error)
latest = cfg.path(cfg.output_dir) / "latest"
assert (latest / "state.json").is_file()

Saved epoch-one checkpoint.


### 3. 恢复并检查 held-out 指标

恢复须保持配置、数据、源码和环境不变。Validation 选择最小指标，包含基线；test 仅报告。

In [5]:
report = run(cfg, resume=latest)
assert report["training"]["epoch"] == cfg.training.epochs
metrics = {"baseline_test": report["baseline_test"]["metrics"],
           "selected_test": report["finetuned_test"]["metrics"],
           "selected_epoch": report["training"]["best_epoch"]}
print(json.dumps(metrics, indent=2))

{
  "baseline_test": {
    "nll": 3.531214952468872,
    "wer": 4.0,
    "cer": 5.625
  },
  "selected_test": {
    "nll": 3.4273252487182617,
    "wer": 1.0,
    "cer": 1.0
  },
  "selected_epoch": 1
}


上述数值只诊断 smoke，不是 benchmark。STT 因插入错误 WER/CER 可大于 1；未分词中文优先 CER。TTS loss 是 teacher forcing 而非自然度评分；基线也可能被选为最佳模型。

### 4. 重载与推理

加载最佳导出包，不依赖原训练数据。不会上传权重或音频。

In [6]:
selected = STTModel.load(report['export'], 'cpu')

In [7]:
transcription = selected.transcribe(rows[-1]["audio"])
print({"reference": rows[-1]["text"], "transcription": transcription})

{'reference': 'pink', 'transcription': ''}


## Checks / 检查

检查原生导出与元数据、重载一致性及离线白名单发布预览。Speaker embedding 是衍生生物特征，只有获得明确授权后才发布。

In [8]:
export = Path(report["export"])
assert (export / "advai_metadata.json").is_file()
reloaded = STTModel.load(export, "cpu")
for name, value in selected.model.state_dict().items():
    assert torch.equal(value, reloaded.model.state_dict()[name])
plan = publish(export, "example/stt-smoke", dry_run=True)
names = [item["path"] for item in plan["files"]]
assert not any(name.endswith((".wav", ".jsonl", ".pt")) for name in names)
print({"reload_equal": True, "publish": plan["status"], "files": len(names)})

{'reload_equal': True, 'publish': 'dry_run', 'files': 9}


## Next Steps / 下一步

阅读[完整指南](../../docs/stt.zh-CN.md)，复制 `configs/stt/hf.yaml` 到 `configs/local/`，固定模型版本并使用自己的授权语音数据，按独立会话划分与保留未见文本。真实配置需目标硬件验证，不要在这些正弦音上训练预训练语音模型。已执行教程不证明 GPU 微调、免费 Colab 显存适配或语音质量。TTS 使用相同未见文本对比基线与最佳模型听感。